In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

class Visualizer:
    @staticmethod
    def plot_combined_figure(
        l_syn_df, l_real_df, s_syn_df, s_real_df,
        l_synthetic_k_df, conv_m_df, T
    ):
        # ===================== preprocess =====================
        for df in [l_syn_df, l_real_df, s_syn_df, s_real_df,
                   l_synthetic_k_df, conv_m_df]:
            df.rename(columns={'algorithm': 'Algorithm'}, inplace=True)
            df['Algorithm'] = df['Algorithm'].replace({'EXP3_M': 'EXP3.M'})

        all_df = pd.concat([
            l_syn_df, l_real_df, s_syn_df, s_real_df,
            l_synthetic_k_df, conv_m_df
        ])
        algorithms = sorted(all_df['Algorithm'].dropna().unique().tolist())

        for df in [l_syn_df, l_real_df, s_syn_df, s_real_df,
                   l_synthetic_k_df, conv_m_df]:
            df['Algorithm'] = pd.Categorical(
                df['Algorithm'], categories=algorithms, ordered=True
            )

        # ===================== figure =====================
        sns.set_theme(style="white", font_scale=3)
        fig, axes = plt.subplots(3, 4, figsize=(32, 14), sharex=False)

        # ===================== helper =====================
        def draw_line(ax, df, k_val):
            sub = df[df["k"] == k_val].copy()

            sub["t_scaled"] = sub["t"] / 1e4
            sub["regret_scaled"] = sub["regret"] / 1e3

            sns.lineplot(
                data=sub,
                x='t_scaled', y='regret_scaled',
                hue='Algorithm',
                hue_order=algorithms,
                estimator='mean',
                errorbar='sd',
                palette='tab10',
                ax=ax,
            )

            ax.set_xlabel("Rounds ($\\times 10^4$)")
            ax.set_ylabel("Regret ($\\times 10^3$)")
            ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
            ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=3))
            ax.get_legend().remove()

        # ===================== line plots =====================
        # Row 1: Synthetic
        draw_line(axes[0][0], l_syn_df, 10)
        draw_line(axes[0][1], s_syn_df, 10)
        draw_line(axes[0][2], l_syn_df, 20)
        draw_line(axes[0][3], s_syn_df, 20)

        # Row 2: Yelp
        draw_line(axes[1][0], l_real_df, 10)
        draw_line(axes[1][1], s_real_df, 10)
        draw_line(axes[1][2], l_real_df, 20)
        draw_line(axes[1][3], s_real_df, 20)

        # ===================== bar plots =====================
        # Regret: 除以 1e3
        bar_k = l_synthetic_k_df.copy()
        bar_k["regret_scaled"] = bar_k["regret"] / 1e3

        sns.barplot(
            data=bar_k,
            x='k', y='regret_scaled',
            hue='Algorithm',
            hue_order=algorithms,
            errorbar=None,
            palette='tab10',
            dodge=True,
            capsize=0.1,
            ax=axes[2][0]
        )
        axes[2][0].set_xlabel("$k$")
        axes[2][0].set_ylabel("Regret ($\\times 10^3$)")
        axes[2][0].yaxis.set_major_locator(ticker.MaxNLocator(nbins=3))
        axes[2][0].get_legend().remove()

        sns.barplot(
            data=bar_k,
            x='k', y='cumulative_time',
            hue='Algorithm',
            hue_order=algorithms,
            errorbar=None,
            palette='tab10',
            dodge=True,
            capsize=0.1,
            ax=axes[2][1]
        )
        axes[2][1].set_xlabel("$k$")
        axes[2][1].set_ylabel("Runtime (sec)")
        axes[2][1].get_legend().remove()

        bar_m = conv_m_df.copy()
        bar_m["regret_scaled"] = bar_m["regret"] / 1e3
        sns.barplot(
            data=bar_m,
            x='m', y='regret_scaled',
            hue='Algorithm',
            hue_order=algorithms,
            errorbar=None,
            palette='tab10',
            dodge=True,
            capsize=0.1,
            ax=axes[2][2]
        )
        axes[2][2].set_xlabel("$m$")
        axes[2][2].set_ylabel("Regret ($\\times 10^3$)")
        axes[2][2].get_legend().remove()

        sns.barplot(
            data=conv_m_df,
            x='m', y='cumulative_time',
            hue='Algorithm',
            hue_order=algorithms,
            errorbar=None,
            palette='tab10',
            dodge=True,
            capsize=0.1,
            ax=axes[2][3]
        )
        axes[2][3].set_xlabel("$m$")
        axes[2][3].set_ylabel("Runtime (sec)")
        axes[2][3].get_legend().remove()

        # ===================== legends =====================
        handles_line, labels_line = axes[0][0].get_legend_handles_labels()
        handles_bar, labels_bar = axes[2][0].get_legend_handles_labels()

        fig.legend(
            handles_line, labels_line,
            loc='upper center',
            bbox_to_anchor=(0.32, 1.05),
            ncol=len(labels_line),
            frameon=False,
            prop={'size': 24}
        )

        fig.legend(
            handles_bar, labels_bar,
            loc='upper center',
            bbox_to_anchor=(0.82, 1.05),
            ncol=len(labels_bar),
            frameon=False,
            prop={'size': 24}
        )

        # ===================== layout finalize =====================
        fig.tight_layout(rect=[0.02, 0.03, 1, 0.98])
        fig.canvas.draw()

        # ===================== save subplots =====================
        renderer = fig.canvas.get_renderer()
        subplot_labels = [f"({i})" for i in range(1, 13)]

        for idx, ax in enumerate(axes.flat):
            bbox = ax.get_tightbbox(renderer).transformed(
                fig.dpi_scale_trans.inverted()
            )
            bbox = bbox.expanded(1.02, 1.15)

            fig.savefig(
                f"split_figs/subplot_{idx+1:02d}{subplot_labels[idx]}.pdf",
                bbox_inches=bbox,
                pad_inches=0.02,
                dpi=600
            )
            fig.savefig(
                f"split_figs/subplot_{idx+1:02d}{subplot_labels[idx]}.png",
                bbox_inches=bbox,
                pad_inches=0.02,
                dpi=600
            )

        # ===================== legend export =====================
        fig_leg1 = plt.figure(figsize=(len(labels_line) * 2.2, 1.8))
        fig_leg1.legend(
            handles_line, labels_line,
            loc="center",
            ncol=len(labels_line),
            frameon=False,
            prop={'size': 24}
        )
        fig_leg1.savefig("split_figs/legend_line.pdf", bbox_inches="tight")
        plt.close(fig_leg1)

        fig_leg2 = plt.figure(figsize=(len(labels_bar) * 2.2, 1.8))
        fig_leg2.legend(
            handles_bar, labels_bar,
            loc="center",
            ncol=len(labels_bar),
            frameon=False,
            prop={'size': 24}
        )
        fig_leg2.savefig("split_figs/legend_bar.pdf", bbox_inches="tight")
        plt.close(fig_leg2)
        

if __name__ == "__main__":
    T = 100000

    # Synthetic line
    l_syn_df = pd.read_csv("./results/l_conv_syn.csv")
    l_syn_df = l_syn_df[l_syn_df['t'] <= T]

    # Synthetic sliding
    s_syn_df = pd.read_csv("./results/s_conv_syn.csv")
    s_syn_df = s_syn_df[s_syn_df['t'] <= T]

    # Real (Yelp) line
    l_real_df = pd.read_csv("./results/l_conv_yelp.csv")
    l_real_df = l_real_df[l_real_df['t'] <= T]

    # Real (Yelp) sliding
    s_real_df = pd.read_csv("./results/s_conv_yelp.csv")
    s_real_df = s_real_df[s_real_df['t'] <= T]

    # Varying k on synthetic
    l_synthetic_k_df = pd.read_csv("./results/l_conv_synthetic_varying_k.csv")
    l_synthetic_k_df = l_synthetic_k_df[
        l_synthetic_k_df['k'].isin([5, 10, 15, 20, 25])
    ]
    idx = l_synthetic_k_df.groupby(
        ['sim_id', 'k', 'algorithm']
    )['t'].idxmax()
    l_synthetic_k_df = l_synthetic_k_df.loc[idx]

    # Varying m
    l_conv_m_df = pd.read_csv("./results/l_conv_synthetic_varying_m.csv")
    l_conv_m_df = l_conv_m_df[l_conv_m_df['m'].isin([20, 25, 30, 35, 40])]
    l_conv_m_df = l_conv_m_df[l_conv_m_df['t'] <= T]
    idx = l_conv_m_df.groupby(
        ['sim_id', 'm', 'algorithm']
    )['t'].idxmax()
    l_conv_m_df = l_conv_m_df.loc[idx]

    Visualizer.plot_combined_figure(
        l_syn_df, l_real_df, s_syn_df, s_real_df,
        l_synthetic_k_df, l_conv_m_df,
        T=T
    )
